In [17]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [18]:
df_ee = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/ee_bn_no_context.csv")

In [19]:
df_ee = df_ee.drop(["Unnamed: 0"], axis = 1)

In [20]:
df_ee

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,12234.211743,26.84,76.05,4.99,3.78,10.10,17.139,-8.790011,15.899159,3.169094,13.51,-3.3,19.4,0.000017,32.835821,55.095409
1,12429.172307,26.13,76.05,5.10,4.00,12.03,18.811,-3.904339,18.436467,-2.025402,8.12,-2.0,19.5,0.000016,46.268657,84.791865
2,10590.021094,23.28,70.50,4.87,4.22,27.41,23.009,4.963764,20.533284,-2.239512,9.51,-1.8,19.7,0.000013,57.142857,356.850436
3,11071.755760,24.03,67.60,4.55,4.53,32.94,24.575,6.420120,19.561652,-0.609832,13.28,-2.7,15.8,0.000017,63.157895,396.929938
4,12612.147903,25.35,71.00,4.39,4.49,22.39,25.515,5.793163,18.294990,0.526226,4.80,-3.3,17.5,0.000017,46.616541,179.744167
5,13674.010506,24.63,73.10,4.46,4.22,20.86,25.586,1.640113,18.024258,-0.123296,7.69,-3.8,17.5,0.000016,15.151515,499.062449
6,14539.662078,24.68,74.10,4.14,4.20,18.70,25.356,2.306589,18.399173,0.273440,4.31,-3.3,18.6,0.000018,18.939394,514.084077
7,15492.493600,24.11,75.00,3.78,4.39,14.98,26.130,3.225526,18.439013,0.975175,6.59,-1.9,21.8,0.000017,33.587786,287.150438
8,15972.915607,23.12,76.70,3.73,4.53,14.34,28.987,3.561763,19.223265,0.119938,-3.07,0.8,21.6,0.000016,22.727273,79.744303
9,16863.602095,23.23,77.00,4.50,4.70,14.13,29.232,3.207492,19.332394,-0.181622,3.77,-0.2,21.7,0.000016,21.969697,146.147423


In [21]:
ee = pd.DataFrame()
ee = df_ee.copy()
ee['EUPC_3'] = ee['EUPC'].shift(3)
ee['CO2C_3'] = ee['CO2C'].shift(3)

In [22]:
ee = ee.dropna()

In [23]:
sm = StructureModel()

In [24]:
sm.add_edges_from([
    ('EUPC_3', 'EBSG'),
    ('QOR', 'EBSG'),
    ('REST', 'EBSG'),
    ('GGFC', 'QOR'),
    ('GGFC', 'LBGD'),
    ('CO2C_3', 'LBGD'),
    ('CO2C_3', 'FDI'),
    ('CO2C_3', 'PC'),
])

In [25]:
sm.edges

OutEdgeView([('EUPC_3', 'EBSG'), ('QOR', 'EBSG'), ('REST', 'EBSG'), ('GGFC', 'QOR'), ('GGFC', 'LBGD'), ('CO2C_3', 'LBGD'), ('CO2C_3', 'FDI'), ('CO2C_3', 'PC')])

In [26]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_ee_1.html")

Graphs/fully_connected_ee_1.html


In [27]:
bn = BayesianNetwork(sm)

In [28]:
import pandas as pd
from causalnex.discretiser import Discretiser

def discretize_dataframe(df, max_buckets=4):
    discretised_df = df.copy()
    
    for col in df.columns:
        # Drop NaN values before processing
        if df[col].isna().any():
            print(f"Warning: {col} has NaN values, filling with median")
            df[col] = df[col].fillna(df[col].median())
        
        unique_vals = df[col].nunique()
        
        if unique_vals <= 1:
            discretised_df[col] = 0
            continue
            
        n_buckets = min(max_buckets, unique_vals)
        
        try:
            d = Discretiser(method="quantile", num_buckets=n_buckets)
            discretised_df[col] = d.fit_transform(df[[col]].values).flatten()
            
        except (ValueError, IndexError):
            discretised_df[col] = df[col].rank(method='dense').astype(int) - 1
    
    # CRITICAL: Ensure all values are integers and no NaNs
    discretised_df = discretised_df.fillna(0).astype(int)
    
    return discretised_df

# Usage:
discretised_ee = discretize_dataframe(ee, max_buckets=4)

# Verify before using with BN
print("Data types:")
print(discretised_ee.dtypes)
print("\nAny NaN values?")
print(discretised_ee.isna().sum())
print("\nValue ranges:")
print(discretised_ee.describe())

# Then use with BN
discretised_ee = discretised_ee.reset_index(drop=True)
bn.fit_node_states(discretised_ee)
baseline_auc = utils.get_avg_auc_all_info(discretised_ee, bn)
print(f"Baseline AUC: {baseline_auc}")

Data types:
GDPC      int64
VABI      int64
EMP       int64
QOA       int64
QOR       int64
YUA       int64
REST      int64
EBSG      int64
GGFC      int64
LBGD      int64
FDI       int64
PC        int64
ARP       int64
CO2C      int64
PAM       int64
EUPC      int64
EUPC_3    int64
CO2C_3    int64
dtype: object

Any NaN values?
GDPC      0
VABI      0
EMP       0
QOA       0
QOR       0
YUA       0
REST      0
EBSG      0
GGFC      0
LBGD      0
FDI       0
PC        0
ARP       0
CO2C      0
PAM       0
EUPC      0
EUPC_3    0
CO2C_3    0
dtype: int64

Value ranges:
            GDPC       VABI        EMP        QOA        QOR        YUA  \
count  13.000000  13.000000  13.000000  13.000000  13.000000  13.000000   
mean    1.615385   1.615385   1.615385   1.769231   1.846154   1.615385   
std     1.192928   1.192928   1.192928   1.300887   1.344504   1.192928   
min     0.000000   0.000000   0.000000   0.000000   0.000000   0.000000   
25%     1.000000   1.000000   1.000000   1.000000 

ValueError: unknown format is not supported

In [ ]:
discretised_ee = discretised_ee.reset_index(drop=True)
bn.fit_node_states(discretised_ee)
baseline_auc = utils.get_avg_auc_all_info(discretised_ee, bn)
print(f"Baseline AUC: {baseline_auc}")

In [ ]:
edges_to_add = [('LV', 'EUPC_3'), ('LV', 'EBSG')]
edges_to_remove = [('EUPC_3', 'EBSG')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [ ]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_ee_1.html")

In [ ]:
discretised_ee['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_ee, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC_3' and 'EBSG': {proposed_auc}")

In [21]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [22]:
df_ee_1 = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/ee_bn_no_context.csv")

In [23]:
df_ee_1 = df_ee_1.drop(["Unnamed: 0"], axis = 1)

In [24]:
df_ee_1

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,12234.211743,26.84,76.05,4.99,3.78,10.10,17.139,-8.790011,15.899159,3.169094,13.51,-3.3,19.4,0.000017,32.835821,55.095409
1,12429.172307,26.13,76.05,5.10,4.00,12.03,18.811,-3.904339,18.436467,-2.025402,8.12,-2.0,19.5,0.000016,46.268657,84.791865
2,10590.021094,23.28,70.50,4.87,4.22,27.41,23.009,4.963764,20.533284,-2.239512,9.51,-1.8,19.7,0.000013,57.142857,356.850436
3,11071.755760,24.03,67.60,4.55,4.53,32.94,24.575,6.420120,19.561652,-0.609832,13.28,-2.7,15.8,0.000017,63.157895,396.929938
4,12612.147903,25.35,71.00,4.39,4.49,22.39,25.515,5.793163,18.294990,0.526226,4.80,-3.3,17.5,0.000017,46.616541,179.744167
5,13674.010506,24.63,73.10,4.46,4.22,20.86,25.586,1.640113,18.024258,-0.123296,7.69,-3.8,17.5,0.000016,15.151515,499.062449
6,14539.662078,24.68,74.10,4.14,4.20,18.70,25.356,2.306589,18.399173,0.273440,4.31,-3.3,18.6,0.000018,18.939394,514.084077
7,15492.493600,24.11,75.00,3.78,4.39,14.98,26.130,3.225526,18.439013,0.975175,6.59,-1.9,21.8,0.000017,33.587786,287.150438
8,15972.915607,23.12,76.70,3.73,4.53,14.34,28.987,3.561763,19.223265,0.119938,-3.07,0.8,21.6,0.000016,22.727273,79.744303
9,16863.602095,23.23,77.00,4.50,4.70,14.13,29.232,3.207492,19.332394,-0.181622,3.77,-0.2,21.7,0.000016,21.969697,146.147423


In [25]:
ee_1 = pd.DataFrame()
ee_1 = df_ee_1.copy()
ee_1['QOR_2'] = ee_1['QOR'].shift(2)
ee_1['VABI_2'] = ee_1['VABI'].shift(2)
ee_1['QOA_1'] = ee_1['QOA'].shift(1)
ee_1['CO2C_3'] = ee_1['CO2C'].shift(3)

In [26]:
ee_1 = ee_1.dropna()

In [27]:
sm1 = StructureModel()

In [28]:
sm1.add_edges_from([
    ('PAM', 'EUPC'),
    ('VABI_2', 'EUPC'),
     ('QOR_2', 'PAM'),
    ('EBSG', 'PAM'),
    ('QOR', 'EBSG'),
    ('REST', 'EBSG'),
    ('GGFC', 'QOR'),
    ('LBGD', 'GGFC'),
    ('QOA_1', 'LBGD'),
#     ('CO2C_3', 'LBGD'),
#     ('CO2C_3', 'FDI'),
#     ('CO2C_3', 'PC')
])

In [29]:
sm1.edges

OutEdgeView([('PAM', 'EUPC'), ('VABI_2', 'EUPC'), ('QOR_2', 'PAM'), ('EBSG', 'PAM'), ('QOR', 'EBSG'), ('REST', 'EBSG'), ('GGFC', 'QOR'), ('LBGD', 'GGFC'), ('QOA_1', 'LBGD')])

In [30]:
viz = plot_structure(
    sm1,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_pl_2.html")

Graphs/fully_connected_pl_2.html


In [31]:
bn1 = BayesianNetwork(sm1)

In [32]:
discretised_ee = pd.DataFrame(index=ee_1.index)

for col in ee_1.columns:
    no_unique = ee_1[col].nunique()
    
    if no_unique <= 1:
        discretised_ee[col] = 0
    else:
        try:
            discretised_ee[col] = pd.qcut(
                ee_1[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_ee[col] = ee_1[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_ee)

Discretised data:
    GDPC  VABI  EMP  QOA  QOR  YUA  REST  EBSG  GGFC  LBGD  FDI  PC  ARP  \
3      0     1    0    1    0    2     0     2     2     0    2   0    0   
4      0     2    0    0    0    2     0     2     0     2    0   0    0   
5      0     2    0    0    0    2     0     0     0     1    1   0    0   
6      0     2    0    0    0    2     0     0     0     2    0   0    0   
7      0     2    0    0    0    1     0     1     0     2    1   0    2   
8      1     0    1    0    0    0     1     1     1     1    0   1    1   
9      1     1    1    1    1    0     1     1     2     1    0   1    1   
10     1     0    1    2    1    0     1     2     1     1    1   2    1   
11     1     1    2    1    1    0     1     1     0     0    0   2    2   
12     2     0    2    1    1    0     2     2     1     2    2   2    1   
13     2     0    1    1    1    1     2     0     2     0    2   1    1   
14     2     0    2    1    1    1     2     0     2     0    2   1   

In [33]:
discretised_ee = discretised_ee.reset_index(drop=True)
bn1.fit_node_states(discretised_ee)
baseline_auc = utils.get_avg_auc_all_info(discretised_ee, bn1)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 6.090302228927612 seconds
Processing fold 1 using 7 cores takes 5.474819898605347 seconds
Processing fold 2 using 7 cores takes 5.375183820724487 seconds
Processing fold 3 using 7 cores takes 5.23789381980896 seconds
Processing fold 4 using 7 cores takes 5.412672996520996 seconds
Baseline AUC: 0.5511111111111111


In [34]:
edges_to_add = [('LV', 'PAM'), ('LV', 'EUPC')]
edges_to_remove = [('PAM', 'EUPC')]

bn_with_lv = copy.deepcopy(bn1)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [35]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_ee_3.html")

Graphs/node_added_ee_3.html


In [36]:
discretised_ee['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_ee, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'PAM' and 'EUPC': {proposed_auc}")

Processing fold 0 using 7 cores takes 6.6890459060668945 seconds
Processing fold 1 using 7 cores takes 6.282586097717285 seconds
Processing fold 2 using 7 cores takes 6.1653289794921875 seconds
Processing fold 3 using 7 cores takes 6.033827066421509 seconds
Processing fold 4 using 7 cores takes 5.891664981842041 seconds
AUC from adding LV between 'PAM' and 'EUPC': 0.5244444444444445


In [37]:
edges_to_add = [('LV', 'VABI_2'), ('LV', 'EUPC')]
edges_to_remove = [('VABI_2', 'EUPC')]

bn_with_lv = copy.deepcopy(bn1)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [38]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_ee_4.html")

Graphs/node_added_ee_4.html


In [39]:
discretised_ee['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_ee, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'VABI_2' and 'EUPC': {proposed_auc}")

Processing fold 0 using 7 cores takes 6.586703062057495 seconds
Processing fold 1 using 7 cores takes 6.011469125747681 seconds
Processing fold 2 using 7 cores takes 5.794328212738037 seconds
Processing fold 3 using 7 cores takes 5.831171035766602 seconds
Processing fold 4 using 7 cores takes 5.8798840045928955 seconds
AUC from adding LV between 'VABI_2' and 'EUPC': 0.66
